# S2F — Few-Shot Segmentation (Colab)

Pipeline: sam encoder → **ProtoFormer** (rgb+topografi, fusi weight-shared) → skip connection.

Semua settingan training ada di **`train.sh`**; notebook hanya:
1. set **path** + `TUNING` + daftar run,
2. sync repo, install deps,
3. siapkan data (PASCAL opsional),
4. jalankan `bash train.sh ...` dengan output **streaming** (stdout+stderr tampil di cell),
5. evaluasi lewat `bash train.sh ... --test-only`.

Dataset default: **landslide** (`3.TRAINING` / `4.VALIDATION` / `5.TESTING`).

## 1. Konfigurasi path, TUNING, daftar run

Hyperparameter training diatur di dict **`TUNING`**: nilainya dikirim sebagai environment
variable ke `train.sh` (`SHOT`, `SIZE`, `EPOCHS`, ...). Set `""` untuk memakai default `train.sh`.
Prioritas: flag CLI di `RUNS`/`EXTRA_ARGS` > `TUNING`/env > default `train.sh`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR    = "/content/drive/MyDrive/project/S2F"
REPO_URL    = "https://github.com/USER/REPO.git"   # WAJIB diisi
REPO_BRANCH = ""   # "" = auto-deteksi branch default (main/master)

# ---- pilih dataset: "landslide" atau "pascal" ----
DATASET = "landslide"

# ---- path dataset landslide (default = struktur Google Drive kamu) ----
TRAIN_ROOT = "/content/drive/MyDrive/project/3.TRAINING"
VAL_ROOT   = "/content/drive/MyDrive/project/4.VALIDATION"
TEST_ROOT  = "/content/drive/MyDrive/project/5.TESTING"
BLACKLIST  = "/content/drive/MyDrive/project/black_list.txt"
HIERA_CKPT = "/content/drive/MyDrive/project/checkpoint/sam2_hiera_large.pt"
OUT_DIR    = "/content/drive/MyDrive/project/runs"

# ---- parameter PASCAL (hanya dipakai kalau DATASET == "pascal") ----
DATA_ROOT_LOCAL = "/content/datasets"
ARCHIVE_DIR     = "/content/drive/MyDrive/project/datasets/_archives"
SPLIT_ROOT      = "/content/drive/MyDrive/project/datasets/splits/pascal"
CACHE_DIR       = "/content/drive/MyDrive/project/datasets/_cache"
FOLDS           = [0, 1, 2, 3]

VOC_DIR  = f"{DATA_ROOT_LOCAL}/VOC2012"
SBD_DIR  = f"{DATA_ROOT_LOCAL}/SBD"
SBD_IMG  = f"{SBD_DIR}/img"
SBD_CLS  = f"{SBD_DIR}/cls"
SBD_PNG  = f"{SBD_DIR}/cls_png"
VOC_IMG  = f"{VOC_DIR}/JPEGImages"
VOC_MASK = f"{VOC_DIR}/SegmentationClass"

# ---- HYPERPARAMETER: dikirim ke train.sh via environment ----
TUNING = {
    "SHOT":               1,
    "SIZE":               352,
    "REDUCE_DIM":         64,
    "EPOCHS":             20,
    "EPISODES_PER_EPOCH": 1000,
    "VAL_EPISODES":       1000,
    "VAL_INTERVAL":       1,
    "BATCH_SIZE":         4,
    "LR":                 "1e-3",
    "WEIGHT_DECAY":       "5e-4",
    "NUM_WORKERS":        4,
    "SEED":               0,
    "AMP_DTYPE":          "auto",   # auto | bf16 | fp16 | none
    "CACHE_DIR":          CACHE_DIR,  # "" = tanpa cache
}

# ---- override tambahan per-run (kosongkan = pakai default train.sh) ----
EXTRA_ARGS = []   # mis: ["--shot", "5", "--epochs", "30"]

# ---- daftar run ----
if DATASET == "landslide":
    RUNS = [
        ["--dataset", "landslide",
         "--train-root", TRAIN_ROOT, "--val-root", VAL_ROOT, "--test-root", TEST_ROOT,
         "--blacklist", BLACKLIST, "--hiera-path", HIERA_CKPT, "--output", OUT_DIR,
         "--feature", "dtm"],
        # contoh run kedua (5-shot, fusi DTM+HILLSHADE):
        # ["--dataset", "landslide",
        #  "--train-root", TRAIN_ROOT, "--val-root", VAL_ROOT, "--test-root", TEST_ROOT,
        #  "--blacklist", BLACKLIST, "--hiera-path", HIERA_CKPT, "--output", OUT_DIR,
        #  "--topo", "DTM", "HILLSHADE", "--shot", "5"],
    ]
    RUN_LABELS = [f"landslide-run{i}" for i in range(len(RUNS))]
else:
    RUNS, RUN_LABELS = [], []
    for f in FOLDS:
        RUNS.append([
            "--dataset", "pascal", "--fold", str(f),
            "--train-img-dir", SBD_IMG, "--train-mask-dir", SBD_PNG,
            "--train-split", f"{SPLIT_ROOT}/fold{f}/train.txt",
            "--val-img-dir", VOC_IMG, "--val-mask-dir", VOC_MASK,
            "--val-split", f"{SPLIT_ROOT}/fold{f}/val.txt",
            "--test-img-dir", VOC_IMG, "--test-mask-dir", VOC_MASK,
            "--test-split", f"{SPLIT_ROOT}/fold{f}/val.txt",
            "--hiera-path", HIERA_CKPT, "--output", OUT_DIR,
            "--cache-dir", CACHE_DIR,
        ])
        RUN_LABELS.append(f"fold{f}")

print(f"config ok | dataset={DATASET} | runs={len(RUNS)}")
print("tuning:", {k: v for k, v in TUNING.items() if v != ""})
for lbl, run in zip(RUN_LABELS, RUNS):
    print(" -", lbl, ":", " ".join(run))

## 1b. Ambil / perbarui kode dari git

Tidak memakai `reset --hard`: kalau ada perubahan lokal, update dilewati supaya editan tidak hilang.

In [ ]:
import os, subprocess

def sh_out(args):
    print("$ " + " ".join(args))
    return subprocess.run(args, text=True, capture_output=True)

def detect_branch():
    if REPO_BRANCH:
        return REPO_BRANCH
    r = sh_out(["git", "-C", REPO_DIR, "symbolic-ref", "--short",
                "refs/remotes/origin/HEAD"])
    if r.returncode == 0 and "/" in r.stdout:
        return r.stdout.strip().split("/", 1)[1]
    r = sh_out(["git", "-C", REPO_DIR, "branch", "--show-current"])
    return r.stdout.strip() or "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.isdir(REPO_DIR) and os.listdir(REPO_DIR):
        raise RuntimeError(f"{REPO_DIR} ada tapi bukan git repo. Kosongkan atau ganti REPO_DIR.")
    r = sh_out(["git", "clone", REPO_URL, REPO_DIR])
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise RuntimeError("git clone gagal, cek REPO_URL")

r = sh_out(["git", "-C", REPO_DIR, "fetch", "origin", "--prune"])
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise RuntimeError("git fetch gagal, cek remote/REPO_URL")

BRANCH = detect_branch()
print("branch:", BRANCH)

dirty = sh_out(["git", "-C", REPO_DIR, "status", "--porcelain"]).stdout.strip()
if dirty:
    print("PERINGATAN: ada perubahan lokal -> update git DILEWATI (tidak ada reset --hard).")
    print(dirty)
else:
    r = sh_out(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"checkout origin/{BRANCH} gagal")

print(sh_out(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout)

## 2. Setup environment
Ekstensi CUDA SAM2 (`_C`) tidak dipakai pipeline ini, jadi tidak perlu di-build.

In [ ]:
!pip -q install hydra-core==1.3.2 omegaconf imageio pysodmetrics tqdm scipy timm opencv-python

In [ ]:
import os, sys
assert os.path.isdir(REPO_DIR), f"REPO_DIR tidak ada: {REPO_DIR}"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

assert os.path.isfile(os.path.join(REPO_DIR, "train.sh")), "train.sh tidak ditemukan"
check_paths = ([TRAIN_ROOT, VAL_ROOT, TEST_ROOT, BLACKLIST, HIERA_CKPT]
               if DATASET == "landslide"
               else [SBD_IMG, VOC_IMG, VOC_MASK, HIERA_CKPT])
for p in check_paths:
    print("OK " if os.path.exists(p) else "MISSING ", p)

## 3. Siapkan data

- **landslide**: data diasumsikan sudah ada di `TRAIN_ROOT` / `VAL_ROOT` / `TEST_ROOT` (Drive). Tiap folder berisi `image/`, `dtm/`, `hillshade/`, `label/` (case-insensitive).
- **pascal**: unduh + ekstrak + konversi SBD + buat split.

In [ ]:
if DATASET == "pascal":
    get_ipython().system('python download_data.py --archive_dir "$ARCHIVE_DIR" --extract_root "$DATA_ROOT_LOCAL" --ckpt_path "$HIERA_CKPT"')
    if not os.path.isdir(SBD_PNG) or len(os.listdir(SBD_PNG)) == 0:
        get_ipython().system('python prepare_sbd.py --cls_dir "$SBD_CLS" --out_dir "$SBD_PNG"')
    else:
        print("SBD_PNG sudah ada, lewati")
    if not os.path.isdir(f"{SPLIT_ROOT}/fold0"):
        get_ipython().system('python make_splits.py --sbd_img_dir "$SBD_IMG" --sbd_mask_dir "$SBD_PNG" --voc_img_dir "$VOC_IMG" --voc_mask_dir "$VOC_MASK" --out_dir "$SPLIT_ROOT"')
    else:
        print("split sudah ada, lewati")
else:
    for p in [TRAIN_ROOT, VAL_ROOT, TEST_ROOT]:
        print("OK " if os.path.isdir(p) else "MISSING ", p)
    print("OK " if os.path.isfile(BLACKLIST) else "MISSING ", BLACKLIST)
    print("catatan: isi tiap split = image/ dtm/ hillshade/ label/ (case-insensitive)")

## 4. Training (satu run per pemanggilan train.sh)

Output **streaming** ke cell: stdout dan stderr `train.sh` + Python terlihat langsung,
tidak ada `capture_output`/`check=True` yang menyembunyikan pesan error.
Log juga disalin ke `/content/logs/<label>.log`.

In [ ]:
import os, shlex

# TUNING -> environment, diwarisi oleh shell yang dipanggil ipython
os.environ.update({k: str(v) for k, v in TUNING.items() if v != ""})

ip = get_ipython()

def run_sh(sh_args, log_path=None):
    cmd = " ".join(["bash", "train.sh", *map(shlex.quote, sh_args)])
    if log_path:
        inner = f"set -o pipefail; {cmd} 2>&1 | tee -a {shlex.quote(log_path)}"
        cmd = f"bash -c {shlex.quote(inner)}"
    print(">>", cmd, flush=True)
    return ip.system(cmd)

os.makedirs("/content/logs", exist_ok=True)

for label, run in zip(RUN_LABELS, RUNS):
    print("=" * 70)
    print(f"TRAIN {label}")
    print("=" * 70)
    rc = run_sh([*run, *EXTRA_ARGS], log_path=f"/content/logs/{label}.log")
    print(f"[{label}] exit code: {rc}")
    if rc != 0:
        raise RuntimeError(f"train.sh gagal untuk {label} (rc={rc}); lihat pesan error di atas")

## 5. Evaluasi (mIoU, FB-IoU) via `train.sh --test-only`

Output tetap streaming; metrik diambil dari log `/content/logs/<label>_test.log`.

In [ ]:
import re
import numpy as np

rows = []
for label, run in zip(RUN_LABELS, RUNS):
    log_path = f"/content/logs/{label}_test.log"
    print("\n" + "=" * 70)
    print(f"TEST {label}")
    print("=" * 70)
    rc = run_sh([*run, *EXTRA_ARGS, "--test-only"], log_path=log_path)
    print(f"[{label}] exit code: {rc}")
    text = open(log_path, encoding="utf-8", errors="replace").read()
    miou = re.findall(r"mIoU\s*:\s*([\d.]+)", text)
    fb = re.findall(r"FB-IoU\s*:\s*([\d.]+)", text)
    if rc == 0 and miou and fb:
        rows.append((label, float(miou[-1]), float(fb[-1])))

print("\n================ HASIL AKHIR ================")
for label, miou, fb in rows:
    print(f"{label}: mIoU {miou:.4f}  FB-IoU {fb:.4f}")
if rows:
    print(f"mean ({len(rows)} run): mIoU {np.mean([r[1] for r in rows]):.4f}  "
          f"FB-IoU {np.mean([r[2] for r in rows]):.4f}")